# PROfit spline factorization: $\Delta\chi^2$ grids for the z-expansion fits

PROfit combines spline nuisance parameters *multiplicatively*, while the z-expansion response of every bin is an exact multivariate quadratic in the standardized PCA coordinates $\eta$ *with cross terms*, so the fitted model is missing the $\eta_i\eta_j$ terms. This notebook builds, for every z-expansion prior and data suite, the exact prediction $\mu^{\rm exact}(\eta)$ and PROfit's factorized one $\mu^{\rm spline}(\eta)$ on grids in $\eta$, turns them into $\Delta\chi^2_{\rm data}=\chi^2(\mu^{\rm spline})-\chi^2(\mu^{\rm exact})$, reads that off inside the posterior credible regions, and writes the grids that `python/scripts/spline_reweighting.py` uses to reweight the stored chains onto the exact model.

Outputs: `figs/spline_factorization/`, `tables/spline_factorization/`.

In [ ]:
from pathlib import Path
import re
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

import awkward as ak
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
import uproot
from IPython.display import display

# Locate ma_zexp/python/scripts (shared style, fit-output locations) and uboone_ngem/src
# (reweighting code) whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
ngem_src = next(
    (parent / 'uboone_ngem' / 'src' for parent in (start, *start.parents)
     if (parent / 'uboone_ngem' / 'src' / 'zexp_reweighting.py').is_file()),
    None,
)
if helper_dir is None or ngem_src is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts and uboone_ngem/src')
for path in (helper_dir, ngem_src):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from postfit_physical_parameters import (
    DATA_ROOT, FIGURE_ROOT, PUBLICATION_RC, SPECS, SUITE_DATA_DIRS, prior_label,
)
import zexp_reweighting as zr
from zexp_reweighting import MA_CCQE_GRID_GEV, axial_form_factor_zexp, complete_zexp_a_values

mpl.rcParams.update(PUBLICATION_RC)
np.set_printoptions(linewidth=160, precision=5, suppress=True)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

REPO_MA_ZEXP = helper_dir.parents[1]
XML_ROOT = REPO_MA_ZEXP / 'xml'
TABLE_DIR = REPO_MA_ZEXP / 'tables' / 'spline_factorization'
FIG_DIR = FIGURE_ROOT / 'spline_factorization'
PROFIT_SRC = Path('/nevis/riverside/share/epelaez/PROfit')
INPUT_FILE = Path('/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root')
TREE_NAME = 'tree'

SUITES = ('nuwro_fit_results', 'asimov_fit_results', 'opendata_fit_results')
SUITE_XML = {'nuwro_fit_results': 'nuwro', 'asimov_fit_results': 'asimov',
             'opendata_fit_results': 'opendata'}
SUITE_LABEL = {'nuwro_fit_results': 'NuWro fake data', 'asimov_fit_results': 'Asimov',
               'opendata_fit_results': 'Open data'}
# Fit outputs are read from SUITE_DATA_DIRS (the production the other notebooks use). Fits
# that do not exist there are looked up in the newer production directory that carries the
# suite's own name; the summary table records which production every chain came from.
FALLBACK_DATA_DIRS = {suite: suite for suite in SUITES}

# Dipole F_A(0) behind the MaCCQE_UBGenie knots as used when the stored z-expansion branches
# were generated (reproduces them to 2e-7; the working tree of uboone_ngem now carries a
# different zr.GENIE_DIPOLE_FA_Q2_ZERO). See notes/spline_factorization.md.
DIPOLE_FA_Q2_ZERO_IN_FILE = -1.2723

GRID_HALF_RANGE = 3.0      # standard grid [-3, 3]
FINE_STEP = 0.5            # 2D grids and pair slices
COARSE_STEP = 1.0          # full 3D / 4D grids
EXTENDED_HALF_RANGE = 10.0 # uniform-prior fits: the XML restrict range, where PROfit extrapolates
CREDIBLE_LEVELS = (0.68, 0.95)
PASS_68, PASS_95 = 0.01, 0.10
CHAIN_BURN_IN, CHAIN_THIN = 0, 1
SAVE_OUTPUTS = True
SAVE_DPI = 600
# The per-fit dchi2 maps are diagnostics with a rasterized pcolormesh on a canvas up to
# 11.5 x 30 inches: at 600 dpi the Agg buffers of the 33 figures peak above 10 GB, so they
# are written at a lower resolution. The appendix figures keep SAVE_DPI.
MAP_DPI = 200

COLOR_GUIDE = '#D55E00'                     # Okabe-Ito, as in the other publication figures
CONTOUR_STYLES = {0.68: '-', 0.95: '--'}


def xml_configuration(suite):
    # The analysis binning, the subchannels and the POT scale are all this notebook needs from
    # the PROfit XMLs; the full configuration is documented in notes/spline_factorization.md.
    text = (XML_ROOT / SUITE_XML[suite] / 'minerva_k6.xml').read_text()
    text = text[text.index('?>') + 2:] if text.lstrip().startswith('<?xml') else text
    text = re.sub(r'&(?!(amp|lt|gt|quot|apos|#\d+);)', '&amp;', text)   # bare '&&' in weight expressions
    root = ET.fromstring('<root>' + text + '</root>')                    # several top-level elements
    channel = root.find('channel')
    bins2d = channel.find('bins2D')
    return (np.array(bins2d.get('edgesx').split(), float), np.array(bins2d.get('edgesy').split(), float),
            [s.get('name') for s in channel.findall('subchannel')],
            float(root.find('detector').get('pot')) / float(root.find('MCFile').get('pot')))


ANALYSIS_EDGES_X, ANALYSIS_EDGES_Y, SUBCHANNELS, _ = xml_configuration('nuwro_fit_results')
POT_SCALE = {suite: xml_configuration(suite)[3] for suite in SUITES}
N_X, N_Y = len(ANALYSIS_EDGES_X) - 1, len(ANALYSIS_EDGES_Y) - 1
N_BINS = N_X * N_Y
N_SUB = len(SUBCHANNELS)
N_BINS_FULL = N_SUB * N_BINS
ZEXP_FITS = [spec for spec in SPECS if spec.prior is not None]

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_FILE.is_file(), INPUT_FILE
print(f'Binning: {N_X} x {N_Y} = {N_BINS} bins x {N_SUB} subchannels -> {N_BINS_FULL} uncollapsed bins; '
      f'POT scales', {SUITE_XML[s]: round(v, 3) for s, v in POT_SCALE.items()})
print('ma_zexp repo:', REPO_MA_ZEXP)
print('fit outputs :', DATA_ROOT, dict(SUITE_DATA_DIRS))
print('PROfit      :', subprocess.run(['git', '-C', str(PROFIT_SRC), 'log', '-1', '--format=%h %ad %s', '--date=short'],
                                      capture_output=True, text=True).stdout.strip())
print('uboone_ngem :', subprocess.run(['git', '-C', str(ngem_src.parent), 'log', '-1', '--format=%h %ad %s', '--date=short'],
                                      capture_output=True, text=True).stdout.strip(),
      '| uncommitted changes in src/zexp_reweighting.py:',
      bool(subprocess.run(['git', '-C', str(ngem_src.parent), 'diff', '--quiet', '--', 'src/zexp_reweighting.py']).returncode))

## Simulation sample and the per-event quadratic model

The simulated overlay component of the PROfit `MCFile` selection, binned exactly as PROfit bins it; the seven `MaCCQE_UBGenie` knots of every event give the per-event $w(F_A)=c_0+c_1F_A+c_2F_A^2$.

In [ ]:
t0 = time.time()
with uproot.open(INPUT_FILE) as root_file:
    tree = root_file[TREE_NAME]
    raw = tree.arrays(['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_true',
                       'afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'GTruth_gQ2', 'non_genie_net_weight',
                       'MaCCQE_UBGenie', 'wc_truth_nuScatType'], library='ak')
print(f'Read {len(raw):,} rows in {time.time() - t0:.0f} s')

mc = ak.to_numpy((raw.isdata == 0) & (raw.isext == 0) & (raw.isdirt == 0) & (raw.isnuwro == 0) & (raw.afro_1mu1p_sel == 1))
events = raw[mc]
log_q2_reco = np.log10(ak.to_numpy(events.afro_1mu1p_Q2).astype(float))
pn_reco = ak.to_numpy(events.afro_1mu1p_Pn).astype(float)
ix = np.searchsorted(ANALYSIS_EDGES_X, log_q2_reco, side='right') - 1
iy = np.searchsorted(ANALYSIS_EDGES_Y, pn_reco, side='right') - 1
in_range = (ix >= 0) & (ix < N_X) & (iy >= 0) & (iy < N_Y)
subchannel = np.where(ak.to_numpy(events.afro_1mu1p_true) == 1, 0, 1)
print(f'Selected simulated events: {len(events):,}; inside the analysis binning: {in_range.sum():,} '
      f'(signal subchannel {np.sum(in_range & (subchannel == 0)):,}, background {np.sum(in_range & (subchannel == 1)):,})')

events = events[in_range]
U_IDX = (subchannel * N_BINS + ix * N_Y + iy)[in_range]          # uncollapsed bin, subchannel-major, x slow
BASE_WEIGHT = ak.to_numpy(events.non_genie_net_weight).astype(float)  # weight_1 (selection) = 1 here; POT scale applied per suite
Q2_TRUE = ak.to_numpy(events.GTruth_gQ2).astype(float)
MA_WEIGHTS = zr._clean_ma_spline_weights(ak.to_numpy(events.MaCCQE_UBGenie).astype(float))
MODE = np.nan_to_num(ak.to_numpy(events.wc_truth_nuScatType), nan=-1).astype(int)
T_COLLAPSE = np.zeros((N_BINS_FULL, N_BINS))
for s in range(N_SUB):
    T_COLLAPSE[s * N_BINS + np.arange(N_BINS), np.arange(N_BINS)] = 1.0


def collapse(v):
    return v.reshape(N_SUB, N_BINS).sum(axis=0)


def prepare_quadratic(q2, weights, dipole_fa_q2_zero):
    # Same least-squares quadratic as zr._prepare_quadratic_fa_splines, with the dipole F_A(0)
    # made explicit so that the value used for the stored branches can be reproduced.
    fa_grid = dipole_fa_q2_zero / (1.0 + q2[:, None] / MA_CCQE_GRID_GEV[None, :] ** 2) ** 2
    center = fa_grid[:, 3]
    scale = np.ptp(fa_grid, axis=1)
    degenerate = ~np.isfinite(scale) | (np.abs(scale) < 1e-14)
    safe_scale = np.where(degenerate, 1.0, scale)
    x = (fa_grid - center[:, None]) / safe_scale[:, None]
    design = np.stack((np.ones_like(x), x, x ** 2), axis=2)
    gram = np.einsum('nki,nkj->nij', design, design)
    rhs = np.einsum('nki,nk->ni', design, weights)
    coefficients = np.zeros((len(q2), 3))
    valid = ~degenerate
    coefficients[valid] = np.linalg.solve(gram[valid], rhs[valid, :, None])[:, :, 0]
    return center, safe_scale, coefficients, degenerate, weights[:, 3]


QUAD_MODEL = prepare_quadratic(Q2_TRUE, MA_WEIGHTS, DIPOLE_FA_Q2_ZERO_IN_FILE)
print(f'Quadratic model: {np.sum(~QUAD_MODEL[3]):,} events with a fitted (c0, c1, c2); {QUAD_MODEL[3].sum()} degenerate (Q2=0).')
print(f'Events with c2 != 0 (respond to F_A): {np.sum(np.abs(QUAD_MODEL[2][:, 2]) > 1e-12):,}; interaction modes of responsive events:',
      dict(zip(*np.unique(MODE[np.abs(QUAD_MODEL[2][:, 1]) > 1e-9], return_counts=True))))


def event_weights(fa, model=QUAD_MODEL):
    # w(F_A) = c0 + c1 x + c2 x^2 with x = (F_A - center)/scale, plus the production code's rule
    # that negative or non-finite weights are reset to one (zr._evaluate_quadratic_fa_splines).
    return zr._evaluate_quadratic_fa_splines(fa, model)

## Priors, PCA directions and the affine map $a(\eta)$

$\Delta a_i$ is the completed coefficient shift (sum rules and $F_A(0)$ re-solved) for one standard deviation along PCA direction $i$; the constraints are linear, so $a(\eta)=a^{\rm CV}+\sum_i\eta_i\Delta a_i$ exactly.

In [ ]:
def pca_shifts(prior):
    covariance = np.asarray(prior.covariance)
    values, vectors = np.linalg.eigh((covariance + covariance.T) / 2)
    order = np.argsort(values)[::-1]
    return vectors[:, order] * np.sqrt(np.clip(values[order], 0, None)), values[order]


def complete(prior, free):
    return complete_zexp_a_values(free, prior.kmax, prior.t0_gev2, t_cut_gev2=prior.t_cut_gev2,
                                  fa_q2_zero=prior.fa_q2_zero)


PRIOR_GROUPS = {}
for spec in ZEXP_FITS:
    prior = spec.prior
    group = PRIOR_GROUPS.setdefault(prior.name, dict(prior=prior, fits=[]))
    group['fits'].append(spec.key)

for name, group in PRIOR_GROUPS.items():
    prior = group['prior']
    shifts, lambdas = pca_shifts(prior)
    n_free = len(prior.free_a_values)
    completed_cv = complete(prior, prior.free_a_values)
    delta_a = np.column_stack([complete(prior, prior.free_a_values + shifts[:, j]) - completed_cv for j in range(n_free)])
    # Exactness of the affine map: a random combination of shifts must equal the linear superposition.
    rng = np.random.default_rng(1)
    for _ in range(5):
        eta = rng.uniform(-3, 3, n_free)
        assert np.allclose(complete(prior, prior.free_a_values + shifts @ eta), completed_cv + delta_a @ eta, atol=1e-9, rtol=0)
    cv_consistency = np.max(np.abs(completed_cv - prior.full_a_values))
    z = ((np.sqrt(prior.t_cut_gev2 + Q2_TRUE) - np.sqrt(prior.t_cut_gev2 - prior.t0_gev2))
         / (np.sqrt(prior.t_cut_gev2 + Q2_TRUE) + np.sqrt(prior.t_cut_gev2 - prior.t0_gev2)))
    powers = z[:, None] ** np.arange(prior.kmax + 1)[None, :]
    group.update(n_free=n_free, shifts=shifts, lambdas=lambdas, sqrt_lambda=np.sqrt(lambdas), delta_a=delta_a,
                 a_cv=np.asarray(prior.full_a_values), fa_cv=powers @ np.asarray(prior.full_a_values),
                 fa_slope=powers @ delta_a, has_uniform_fit=any(s.uniform_prior for s in ZEXP_FITS if s.key in group['fits']),
                 cv_consistency=cv_consistency)
# The published central coefficients are rounded to 8 decimals, so the stored full_a_values satisfy the
# constraints only to ~1e-6 (kmax=7 priors). The sigma=0 knot uses full_a_values, the other knots
# complete(free + sigma*shift): the stored knot set is affine in eta up to this rounding, which shifts
# F_A by ~1e-6, far below the float32 precision of the stored branches.
worst_cv = max(g['cv_consistency'] for g in PRIOR_GROUPS.values())
assert worst_cv < 1e-5, 'a prior central value violates its own constraints'
for name, group in PRIOR_GROUPS.items():
    print(f"{name:28s} kmax {group['prior'].kmax}, {group['n_free']} PCA parameters, sqrt(lambda) "
          f"{np.array2string(group['sqrt_lambda'], precision=4)}: {', '.join(group['fits'])}")
print(f'Affine map a(eta) verified at random eta; max |full_a - completed(free)| = {worst_cv:.1e}.')

## Exact prediction, PROfit spline reimplementation and the $\chi^2$

`mu_exact` sums the per-event quadratic at $F_A(\eta)$ into the 144 uncollapsed bins. `build_profit_splines` and `eval_profit_spline` reproduce `PROsyst::FillSpline` and `PROsyst::GetSplineShift` (CAFAna Hermite construction, `force_0_cv`, no clamping outside the knots), and `mu_spline` combines them multiplicatively as `GetSplineShiftedSpectrum` does. `chi2_data` is PROfit's Neyman $\chi^2$ data term.

In [ ]:
KNOTS = np.arange(-3.0, 4.0)
N_SEGMENTS = len(KNOTS) - 1


def mu_exact(group, eta, clean_like_profit=False):
    fa = group['fa_cv'] + group['fa_slope'] @ np.asarray(eta, float)
    w = event_weights(fa)
    if clean_like_profit:  # PROcreate.cxx: NaN/inf or > 30 universe weights are reset to 1
        w = np.where(np.isfinite(w) & (w <= 30), w, 1.0)
    return np.bincount(U_IDX, weights=BASE_WEIGHT * w, minlength=N_BINS_FULL)


def build_profit_splines(ratios):
    # ratios: (7 knots, n_bins) -> coefficients (n_bins, 6 segments, 4); segment s starts at knot s-3,
    # polynomial in x = eta - knot (unit knot spacing). Transcribed from PROsyst::FillSpline.
    y = ratios
    n = y.shape[1]
    coeffs = np.zeros((n, N_SEGMENTS, 4))
    y1, y2, y3 = y[0], y[1], y[2]
    s = (y3 - y1) / 2
    coeffs[:, 0, :] = np.stack([y1, -2 * y1 + 2 * y2 - s, y1 - y2 + s, np.zeros(n)], axis=1)
    for i in range(1, len(KNOTS) - 2):
        y0, y1, y2, y3 = y[i - 1], y[i], y[i + 1], y[i + 2]
        m1, m2 = (y2 - y0) / 2, (y3 - y1) / 2
        coeffs[:, i, :] = np.stack([y1, m1, -3 * y1 + 3 * y2 - 2 * m1 - m2, 2 * y1 - 2 * y2 + m1 + m2], axis=1)
    y4, y5, y6 = y[-3], y[-2], y[-1]
    s = (y6 - y4) / 2
    coeffs[:, -1, :] = np.stack([y5, s, -y5 + y6 - s, np.zeros(n)], axis=1)
    return coeffs


def eval_profit_spline(coeffs, eta):
    # std::upper_bound on the segment knots, step back one: last segment whose knot <= eta;
    # eta < -3 uses the first segment (x < 0) and eta > 3 the last one (x > 1), i.e. extrapolation.
    segment = int(np.clip(np.floor(eta) + 3, 0, N_SEGMENTS - 1))
    x = eta - KNOTS[segment]
    c = coeffs[:, segment, :]
    return c[:, 0] + x * (c[:, 1] + x * (c[:, 2] + x * c[:, 3]))


for name, group in PRIOR_GROUPS.items():
    cv = mu_exact(group, np.zeros(group['n_free']))
    group['cv'] = cv
    splines = []
    n_reset = 0
    for j in range(group['n_free']):
        knots = []
        for k in KNOTS:
            eta = np.zeros(group['n_free'])
            eta[j] = k
            w_raw = event_weights(group['fa_cv'] + group['fa_slope'] @ eta)
            n_reset += np.sum(~(np.isfinite(w_raw) & (w_raw <= 30)))
            knots.append(mu_exact(group, eta, clean_like_profit=True))
        knots = np.array(knots)
        with np.errstate(invalid='ignore', divide='ignore'):
            ratios = np.where(cv > 0, knots / cv, 1.0)          # force_0_cv: ratio to the eta=0 knot
        splines.append(build_profit_splines(ratios))
    group['splines'] = splines
    group['n_reset_universe_weights'] = n_reset
    print(f'{name:28s} CV total (unit POT scale) {cv.sum():9.2f}; empty uncollapsed bins {np.sum(cv == 0)}; '
          f'universe weights PROfit would reset to 1: {n_reset}')


def mu_spline(group, eta):
    weight = np.ones(N_BINS_FULL)
    for j, coeffs in enumerate(group['splines']):
        weight *= eval_profit_spline(coeffs, float(eta[j]))
    return group['cv'] * weight

In [ ]:
F_FALLBACK = {'lqcd_k7': 'lqcd_k6', 'minerva_lqcd_k7': 'minerva_lqcd_k6'}


def locate(spec_key, suite, pattern):
    for production in (SUITE_DATA_DIRS[suite], FALLBACK_DATA_DIRS[suite]):
        matches = sorted((DATA_ROOT / production / spec_key).glob(pattern))
        if len(matches) == 1:
            return matches[0], production
    return None, None


def load_fractional_covariance(spec_key, suite):
    for candidate in (spec_key, F_FALLBACK.get(spec_key, spec_key), 'minerva_k6'):
        for production in (SUITE_DATA_DIRS[suite], FALLBACK_DATA_DIRS[suite]):
            matches = sorted((DATA_ROOT / production / candidate).glob('*_v1_PROplot.root'))
            if len(matches) != 1:
                continue
            with uproot.open(matches[0]) as f:
                if 'Covariance' not in f:
                    continue
                F = f['Covariance/total_frac_cov'].values().astype(float)
                parts = {k.split(';')[0]: v.values() for k, v in f['Covariance'].items(recursive=False)
                         if k.split(';')[0].endswith('_cov') and not k.startswith('total') and not k.startswith('collapsed')}
            total = sum(v for v in parts.values() if v.shape == F.shape)
            assert np.max(np.abs(total - F)) < 1e-4 * np.abs(F).max(), 'total_frac_cov is not the sum of the stored components'
            assert 'MCStat_cov' in parts
            return F, f'{production}/{candidate}'
    raise FileNotFoundError(f'no fractional covariance for {spec_key} in {suite}')


FIT_OUTPUTS = {}
for suite in SUITES:
    for spec in ZEXP_FITS:
        profile_root, production = locate(spec.key, suite, '*_v1_PROfile.root')
        if profile_root is None:
            FIT_OUTPUTS[(spec.key, suite)] = None
            continue
        with uproot.open(profile_root) as f:
            d = f['nu_uBooNE_numuCC1p_data2d'].values().astype(float).reshape(-1)
            cv_profit = f['nu_uBooNE_numuCC1p_cv2d'].values().astype(float).reshape(-1)
            chains = [k.split(';')[0] for k in f.keys() if k.split(';')[0].endswith('_mcmc_chain')]
            assert len(chains) == 1, chains
            tree = f[chains[0]]
            samples = np.column_stack([tree[b].array(library='np') for b in spec.prior.variation_branches])[CHAIN_BURN_IN::CHAIN_THIN]
        F, f_source = load_fractional_covariance(spec.key, suite)
        FIT_OUTPUTS[(spec.key, suite)] = dict(d=d, cv_profit=cv_profit, samples=samples.astype(float), F=F,
                                              chain_source=production, F_source=f_source, profile_root=profile_root)

summary_inputs = pd.DataFrame([
    {'fit': key, 'suite': SUITE_XML[suite], 'chain production': out['chain_source'], 'n chain samples': len(out['samples']),
     'F source': out['F_source'], 'sum d': out['d'].sum(), 'empty data bins': int(np.sum(out['d'] == 0)),
     'PROfit CV total': out['cv_profit'].sum()}
    for (key, suite), out in FIT_OUTPUTS.items() if out is not None
])
display(summary_inputs)
print('Fits without any PROfit output:', sorted({k for (k, s), v in FIT_OUTPUTS.items() if v is None}))

In [ ]:
DATA_SOURCE = 'profit_output'   # or 'current_cv': Asimov data = current-file central value (see Step 0 discrepancies)


def data_vector(spec_key, suite, group):
    out = FIT_OUTPUTS[(spec_key, suite)]
    if suite == 'asimov_fit_results' and DATA_SOURCE == 'current_cv':
        return collapse(group['cv']) * POT_SCALE[suite]
    return out['d']


def collapsed_syst_cov(F, mu_full):
    S = mu_full[:, None] * T_COLLAPSE            # diag(mu) T, 144 x 72
    return S.T @ F @ S


def chi2_data(mu_full, d, F, active, M_syst=None):
    # PROcovariance::operator(): Neyman stat term diag(d) on bins with d > 0, plus the collapsed,
    # prediction-scaled systematic covariance; no pull term here.
    if M_syst is None:
        M_syst = collapsed_syst_cov(F, mu_full)
    delta = (collapse(mu_full) - d)[active]
    M = np.diag(d[active]) + M_syst[np.ix_(active, active)]
    return float(delta @ np.linalg.solve(M, delta))


def compare_at(group, eta, d, F, active, pot_scale):
    mu_e = mu_exact(group, eta) * pot_scale
    mu_s = mu_spline(group, eta) * pot_scale
    ce, cs = collapse(mu_e), collapse(mu_s)
    frac = np.abs(cs - ce) / np.where(ce > 0, ce, np.inf)
    M_e = collapsed_syst_cov(F, mu_e)
    chi2_e = chi2_data(mu_e, d, F, active, M_e)
    chi2_s_own = chi2_data(mu_s, d, F, active)
    chi2_s_fixed = chi2_data(mu_s, d, F, active, M_e)
    return dict(chi2_exact=chi2_e, dchi2_b1=chi2_s_own - chi2_e, dchi2_b2=chi2_s_fixed - chi2_e,
                max_frac=frac.max(), argmax_frac=int(np.argmax(frac)),
                max_frac_full=np.max(np.abs(mu_s - mu_e) / np.where(mu_e > 0, mu_e, np.inf)))

## Grids of exact versus factorized predictions

Two free parameters: $13\times13$ on $[-3,3]^2$ (step 0.5). Three or four: the full space with step 1.0 plus a step-0.5 slice for every parameter pair. Priors with a uniform-prior fit also get the extended $[-10,10]^2$ grid, where PROfit extrapolates.

In [ ]:
def grid_nodes(n_free, half_range, step):
    axis = np.round(np.arange(-half_range, half_range + step / 2, step), 6)
    mesh = np.meshgrid(*([axis] * n_free), indexing='ij')
    return axis, np.column_stack([m.ravel() for m in mesh])


def pair_slice_nodes(n_free, i, j, half_range, step):
    axis, pair = grid_nodes(2, half_range, step)
    nodes = np.zeros((len(pair), n_free))
    nodes[:, i], nodes[:, j] = pair[:, 0], pair[:, 1]
    return axis, nodes


GRIDS = {}   # (prior name, grid id) -> dict(nodes, axis, pair, kind)
for name, group in PRIOR_GROUPS.items():
    n = group['n_free']
    if n == 2:
        axis, nodes = grid_nodes(2, GRID_HALF_RANGE, FINE_STEP)
        GRIDS[(name, 'full')] = dict(nodes=nodes, axis=axis, pair=(0, 1), kind='full-2D')
    else:
        axis, nodes = grid_nodes(n, GRID_HALF_RANGE, COARSE_STEP)
        GRIDS[(name, 'full')] = dict(nodes=nodes, axis=axis, pair=None, kind=f'full-{n}D-coarse')
        for i in range(n):
            for j in range(i + 1, n):
                axis, nodes = pair_slice_nodes(n, i, j, GRID_HALF_RANGE, FINE_STEP)
                GRIDS[(name, f'pair{i}{j}')] = dict(nodes=nodes, axis=axis, pair=(i, j), kind='pair-slice')
    if group['has_uniform_fit']:
        axis, nodes = grid_nodes(n, EXTENDED_HALF_RANGE, FINE_STEP)
        GRIDS[(name, 'extended')] = dict(nodes=nodes, axis=axis, pair=(0, 1) if n == 2 else None, kind='extended-2D')

t0 = time.time()
RESULTS = {}   # (prior name, grid id, suite) -> DataFrame with one row per node
for (name, grid_id), grid in GRIDS.items():
    group = PRIOR_GROUPS[name]
    for suite in SUITES:
        representative = next((k for k in group['fits'] if FIT_OUTPUTS.get((k, suite)) is not None), None)
        if representative is None:
            continue
        out = FIT_OUTPUTS[(representative, suite)]
        d, F = data_vector(representative, suite, group), out['F']
        active = d > 0
        pot = POT_SCALE[suite]
        records = []
        for eta in grid['nodes']:
            records.append({**{f'eta{i + 1}': eta[i] for i in range(group['n_free'])},
                            **compare_at(group, eta, d, F, active, pot)})
        RESULTS[(name, grid_id, suite)] = pd.DataFrame(records)
print(f'{sum(len(v) for v in RESULTS.values()):,} (node, suite) evaluations in {time.time() - t0:.0f} s')

## Credible regions from the chains and the summary table

Highest-density 68% and 95% regions from the stored MCMC chains decide which grid nodes count. The table is written to `tables/spline_factorization/spline_factorization_summary.csv`.

In [ ]:
def smoothed_density_2d(samples_2d, half_range, mesh_step=0.1, sigma=0.2):
    edges = np.arange(-half_range - mesh_step / 2, half_range + mesh_step, mesh_step)
    hist, _, _ = np.histogram2d(samples_2d[:, 0], samples_2d[:, 1], bins=(edges, edges))
    radius = int(np.ceil(4 * sigma / mesh_step))
    offsets = np.arange(-radius, radius + 1) * mesh_step
    kernel = np.exp(-0.5 * (offsets / sigma) ** 2)
    kernel /= kernel.sum()
    smooth = np.apply_along_axis(lambda v: np.convolve(v, kernel, mode='same'), 0, hist)
    smooth = np.apply_along_axis(lambda v: np.convolve(v, kernel, mode='same'), 1, smooth)
    centers = (edges[:-1] + edges[1:]) / 2
    return centers, smooth / smooth.sum()


def hpd_thresholds(density, levels=CREDIBLE_LEVELS):
    flat = np.sort(density.ravel())[::-1]
    cumulative = np.cumsum(flat)
    return {level: flat[min(np.searchsorted(cumulative, level), len(flat) - 1)] for level in levels}


def membership_2d(nodes_2d, centers, density, thresholds):
    ii = np.rint((nodes_2d[:, 0] - centers[0]) / (centers[1] - centers[0])).astype(int)
    jj = np.rint((nodes_2d[:, 1] - centers[0]) / (centers[1] - centers[0])).astype(int)
    inside = (ii >= 0) & (ii < len(centers)) & (jj >= 0) & (jj < len(centers))
    value = np.zeros(len(nodes_2d))
    value[inside] = density[ii[inside], jj[inside]]
    return {level: value >= threshold for level, threshold in thresholds.items()}


def membership_nd(nodes, samples, half_range, step):
    edges = np.arange(-half_range - step / 2, half_range + step, step)
    hist, _ = np.histogramdd(samples, bins=[edges] * samples.shape[1])
    density = hist / hist.sum()
    thresholds = hpd_thresholds(density)
    idx = tuple(np.rint((nodes[:, k] + half_range) / step).astype(int) for k in range(samples.shape[1]))
    value = density[idx]
    return {level: value >= threshold for level, threshold in thresholds.items()}


def region_maxima(frame, masks):
    out = {}
    for label, mask in [('68', masks[0.68]), ('95', masks[0.95]), ('grid', np.ones(len(frame), bool))]:
        sub = frame[mask]
        if len(sub) == 0:
            for col in ('dchi2_b1', 'dchi2_b2', 'max_frac'):
                out[f'{col}|{label}'] = np.nan
            out[f'n nodes|{label}'] = 0
            continue
        for col in ('dchi2_b1', 'dchi2_b2'):
            out[f'{col}|{label}'] = sub[col].abs().max()
        out[f'max_frac|{label}'] = sub['max_frac'].max()
        out[f'n nodes|{label}'] = len(sub)
    worst = frame.iloc[frame['dchi2_b1'].abs().argmax()]
    out['grid max |dchi2 b1| at'] = '(' + ', '.join(f'{worst[c]:+.1f}' for c in frame.columns if c.startswith('eta')) + ')'
    return out


SUMMARY_ROWS, REGION_MASKS = [], {}
for spec in ZEXP_FITS:
    group = PRIOR_GROUPS[spec.prior.name]
    n = group['n_free']
    for suite in SUITES:
        out = FIT_OUTPUTS.get((spec.key, suite))
        grid_ids = [g for (name, g) in GRIDS if name == spec.prior.name]
        for grid_id in grid_ids:
            grid = GRIDS[(spec.prior.name, grid_id)]
            frame = RESULTS.get((spec.prior.name, grid_id, suite))
            if frame is None:
                continue
            if grid_id == 'extended' and not spec.uniform_prior:
                continue
            row = {'fit': spec.key, 'prior': prior_label(spec), 'kmax': spec.prior.kmax, 'n free': n,
                   'suite': SUITE_XML[suite], 'grid': grid['kind'] + ('' if grid['pair'] is None or grid['kind'] != 'pair-slice' else f' (eta{grid["pair"][0] + 1}, eta{grid["pair"][1] + 1})'),
                   'grid half-range': float(grid['axis'].max()),
                   'chain': out['chain_source'] if out else 'none', 'n samples': len(out['samples']) if out else 0,
                   'F source': out['F_source'] if out else '-'}
            if out is None:
                masks = {0.68: np.zeros(len(frame), bool), 0.95: np.zeros(len(frame), bool)}
                row['region'] = 'no chain'
            else:
                samples = out['samples']
                if grid['pair'] is not None:
                    i, j = grid['pair']
                    centers, density = smoothed_density_2d(samples[:, [i, j]], max(float(grid['axis'].max()), np.abs(samples).max()) + 1)
                    masks = membership_2d(frame[[f'eta{i + 1}', f'eta{j + 1}']].to_numpy(), centers, density, hpd_thresholds(density))
                    row['region'] = 'marginal 2D HPD' if n > 2 else '2D HPD'
                else:
                    masks = membership_nd(frame[[f'eta{k + 1}' for k in range(n)]].to_numpy(), samples, float(grid['axis'].max()), COARSE_STEP)
                    row['region'] = f'{n}D HPD (unit cells)'
                row['chain outside |eta|<=3 (any)'] = np.mean(np.any(np.abs(samples) > 3, axis=1))
                for k in range(n):
                    row[f'chain outside |eta{k + 1}|>3'] = np.mean(np.abs(samples[:, k]) > 3)
                row['chain max |eta|'] = np.abs(samples).max()
            REGION_MASKS[(spec.key, grid_id, suite)] = masks
            row.update(region_maxima(frame, masks))
            # Effect on the result: the fit sampled exp(-chi2_spline/2); reweighting every chain sample by
            # exp(+dchi2/2) (dchi2 interpolated from the grid) recovers the exact-model posterior. Report the
            # shift of the posterior mean in units of the posterior width and the change of the width.
            if out is not None and grid['pair'] is not None and grid_id in ('full', 'extended') and n == 2 or (out is not None and grid['pair'] is None):
                axis = grid['axis']
                shape = (len(axis),) * n
                dchi2_grid = frame.sort_values([f'eta{k + 1}' for k in range(n)])['dchi2_b1'].to_numpy().reshape(shape)
                interpolator = RegularGridInterpolator([axis] * n, dchi2_grid, method='linear', bounds_error=False, fill_value=None)
                clipped = np.clip(out['samples'], axis[0], axis[-1])
                dchi2_at_samples = interpolator(clipped)
                log_w = 0.5 * dchi2_at_samples
                w = np.exp(log_w - log_w.max())
                w /= w.sum()
                mean0, std0 = out['samples'].mean(axis=0), out['samples'].std(axis=0)
                mean1 = (w[:, None] * out['samples']).sum(axis=0)
                std1 = np.sqrt((w[:, None] * (out['samples'] - mean1) ** 2).sum(axis=0))
                row['max |mean shift| / sigma'] = np.max(np.abs(mean1 - mean0) / std0)
                row['max |sigma ratio - 1|'] = np.max(np.abs(std1 / std0 - 1))
                row['chain samples clipped to grid'] = np.mean(np.any(np.abs(out['samples']) > axis[-1], axis=1))
            else:
                row['max |mean shift| / sigma'] = np.nan
                row['max |sigma ratio - 1|'] = np.nan
            row['chi2_exact at eta=0'] = frame.loc[(frame[[c for c in frame.columns if c.startswith('eta')]] == 0).all(axis=1), 'chi2_exact'].iloc[0]
            SUMMARY_ROWS.append(row)
SUMMARY = pd.DataFrame(SUMMARY_ROWS)
SUMMARY['pass 68 (b1)'] = SUMMARY['dchi2_b1|68'] <= PASS_68
SUMMARY['pass 95 (b1)'] = SUMMARY['dchi2_b1|95'] <= PASS_95
if SAVE_OUTPUTS:
    SUMMARY.to_csv(TABLE_DIR / 'spline_factorization_summary.csv', index=False)
    print('Saved:', TABLE_DIR / 'spline_factorization_summary.csv')

show_cols = ['fit', 'suite', 'grid', 'region', 'chain', 'dchi2_b1|68', 'dchi2_b2|68', 'dchi2_b1|95', 'dchi2_b2|95',
             'dchi2_b1|grid', 'dchi2_b2|grid', 'max_frac|68', 'max_frac|95', 'max_frac|grid', 'grid max |dchi2 b1| at',
             'chain outside |eta|<=3 (any)', 'max |mean shift| / sigma', 'max |sigma ratio - 1|', 'pass 68 (b1)', 'pass 95 (b1)']
fmt = ({c: '{:.2e}' for c in show_cols if '|' in c and 'at' not in c and 'chain' not in c and 'shift' not in c and 'ratio' not in c}
       | {'chain outside |eta|<=3 (any)': '{:.3f}', 'max |mean shift| / sigma': '{:.3f}', 'max |sigma ratio - 1|': '{:.3f}'})
# The pair slices of the kmax = 7, 8 priors are in the CSV; shown here are the full-space grids
# and the extended grids of the uniform-prior fits.
print('\n=== Standard grids ([-3, 3]) and full-space regions ===')
display(SUMMARY[SUMMARY['grid'].str.startswith('full')][show_cols].style.format(fmt, na_rep='-'))
print('\n=== Extended grids for the uniform-prior fits ===')
display(SUMMARY[SUMMARY['grid'].str.startswith('extended')][show_cols].style.format(fmt, na_rep='-'))

In [ ]:
def flag(row):
    return (row['dchi2_b1|68'] > PASS_68) or (row['dchi2_b1|95'] > PASS_95)


judged = SUMMARY[(SUMMARY['region'] != 'no chain') & SUMMARY['dchi2_b1|95'].notna()]
failing = judged[judged.apply(flag, axis=1)]
print(f'Criterion |dchi2| <= {PASS_68} (68%) and <= {PASS_95} (95%), data term only, definition b1 '
      f'(each spectrum with its own M_syst): {len(judged) - len(failing)} of {len(judged)} '
      f'(fit, suite, grid) combinations pass. Failing combinations:')
for _, row in failing.iterrows():
    print(f"  {row['fit']:28s} {row['suite']:9s} {row['grid']:22s} 68%: {row['dchi2_b1|68']:.2e}  "
          f"95%: {row['dchi2_b1|95']:.2e}  grid: {row['dchi2_b1|grid']:.2e} at {row['grid max |dchi2 b1| at']}  "
          f"posterior-mean shift: {row['max |mean shift| / sigma']:.3f} sigma")
gaussian = judged[judged['grid'].str.startswith('full') & ~judged['fit'].str.contains('uniform')]
print(f"\nGaussian-prior fits, full grids: worst 68% |dchi2| = {gaussian['dchi2_b1|68'].max():.2e}, "
      f"worst 95% = {gaussian['dchi2_b1|95'].max():.2e}, worst over [-3,3] = {gaussian['dchi2_b1|grid'].max():.2e}; "
      f"largest per-bin fractional difference over [-3,3] = {gaussian['max_frac|grid'].max():.2e}; "
      f"largest posterior-mean shift = {gaussian['max |mean shift| / sigma'].max():.3f} sigma")

## Maps of $\Delta\chi^2$ with the credible contours

One figure per fit and suite ($\Delta\chi^2$ and the largest per-bin fractional difference, with the 68% and 95% contours), written to `figs/spline_factorization/<suite>/` without being displayed.

In [ ]:
def draw_contours(ax, samples_2d, half_range, color='black'):
    centers, density = smoothed_density_2d(samples_2d, half_range + 1)
    thresholds = hpd_thresholds(density)
    for level in CREDIBLE_LEVELS:
        ax.contour(centers, centers, density.T, levels=[thresholds[level]], colors=color,
                   linestyles=CONTOUR_STYLES[level], linewidths=1.3, zorder=5)
    ax.plot(*samples_2d.mean(axis=0), marker='+', color=COLOR_GUIDE, markersize=10, markeredgewidth=1.6, zorder=6)


def map_panel(ax, frame, axis, i, j, column, label, symmetric=True, cmap='RdBu_r'):
    values = frame.pivot(index=f'eta{j + 1}', columns=f'eta{i + 1}', values=column).reindex(index=axis, columns=axis).to_numpy()
    step = axis[1] - axis[0]
    edges = np.r_[axis - step / 2, axis[-1] + step / 2]
    if symmetric:
        vmax = max(np.nanmax(np.abs(values)), 1e-12)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        mesh = ax.pcolormesh(edges, edges, values, cmap=cmap, norm=norm, rasterized=True, zorder=1)
    else:
        mesh = ax.pcolormesh(edges, edges, values, cmap='viridis', rasterized=True, zorder=1)
    ax.axhline(0, color='0.5', lw=0.6, zorder=2)
    ax.axvline(0, color='0.5', lw=0.6, zorder=2)
    ax.set_xlim(edges[0], edges[-1])
    ax.set_ylim(edges[0], edges[-1])
    ax.set_xlabel(rf'$\eta_{i + 1}$')
    ax.set_ylabel(rf'$\eta_{j + 1}$')
    ax.set_aspect('equal')
    cbar = plt.colorbar(mesh, ax=ax, pad=0.02, fraction=0.046)
    cbar.set_label(label)
    return values


def contour_legend(ax, loc='upper left'):
    handles = [Line2D([], [], color='black', ls=CONTOUR_STYLES[0.68], lw=1.3, label='68% credible region'),
               Line2D([], [], color='black', ls=CONTOUR_STYLES[0.95], lw=1.3, label='95% credible region'),
               Line2D([], [], color=COLOR_GUIDE, marker='+', ls='none', markersize=9, markeredgewidth=1.6, label='posterior mean')]
    ax.legend(handles=handles, loc=loc, fontsize=9, frameon=True, framealpha=0.85)


n_saved = 0
for spec in ZEXP_FITS:
    group = PRIOR_GROUPS[spec.prior.name]
    n = group['n_free']
    for suite in SUITES:
        out = FIT_OUTPUTS.get((spec.key, suite))
        grid_ids = [g for (name, g) in GRIDS if name == spec.prior.name and g != 'full' or (name == spec.prior.name and g == 'full' and n == 2)]
        if spec.uniform_prior is False:
            grid_ids = [g for g in grid_ids if g != 'extended']
        panels = [(g, GRIDS[(spec.prior.name, g)]) for g in grid_ids if RESULTS.get((spec.prior.name, g, suite)) is not None]
        if not panels:
            continue
        n_panels = len(panels)
        fig, axes = plt.subplots(n_panels, 2, figsize=(11.5, 5.0 * n_panels), squeeze=False, constrained_layout=True)
        for row_axes, (grid_id, grid) in zip(axes, panels):
            frame = RESULTS[(spec.prior.name, grid_id, suite)]
            i, j = grid['pair']
            for ax, column, label, symmetric in [(row_axes[0], 'dchi2_b1', r'$\Delta\chi^2_{\rm data} = \chi^2(\mu^{\rm spline}) - \chi^2(\mu^{\rm exact})$', True),
                                                 (row_axes[1], 'max_frac', r'$\max_i\,|\mu^{\rm spline}_i - \mu^{\rm exact}_i| / \mu^{\rm exact}_i$', False)]:
                map_panel(ax, frame, grid['axis'], i, j, column, label, symmetric)
                if out is not None:
                    draw_contours(ax, out['samples'][:, [i, j]], float(grid['axis'].max()))
                if n > 2:
                    ax.set_title(f'other $\\eta$ at 0' + ('' if grid_id != 'full' else ''), fontsize=11)
            if out is not None:
                contour_legend(row_axes[0])
        fig.suptitle(f'{spec.key} | {SUITE_LABEL[suite]} | {prior_label(spec)}' + ('' if out else ' | no chain available'), fontsize=13)
        if SAVE_OUTPUTS:
            outdir = FIG_DIR / suite
            outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / f'{spec.key}_dchi2_map.pdf', dpi=MAP_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
            n_saved += 1
        plt.close(fig)   # saved, not displayed
print(f'{n_saved} dchi2 maps written below {FIG_DIR}')

## Summary figures for the PRD appendix

The worst Gaussian-prior case and the worst uniform-prior case, by $|\Delta\chi^2|$ inside the 95% credible region.

In [ ]:
def summary_figure(spec_key, suite, grid_id, stem, half_range=None):
    spec = next(s for s in ZEXP_FITS if s.key == spec_key)
    group = PRIOR_GROUPS[spec.prior.name]
    grid = GRIDS[(spec.prior.name, grid_id)]
    frame = RESULTS[(spec.prior.name, grid_id, suite)]
    out = FIT_OUTPUTS[(spec_key, suite)]
    i, j = grid['pair']
    with mpl.rc_context(PUBLICATION_RC):
        fig, ax = plt.subplots(figsize=(7.0, 5.8), constrained_layout=True)
        values = frame.pivot(index=f'eta{j + 1}', columns=f'eta{i + 1}', values='dchi2_b1').reindex(index=grid['axis'], columns=grid['axis']).to_numpy()
        step = grid['axis'][1] - grid['axis'][0]
        edges = np.r_[grid['axis'] - step / 2, grid['axis'][-1] + step / 2]
        vmax = max(np.nanmax(np.abs(values)), 1e-12)
        mesh = ax.pcolormesh(edges, edges, values, cmap='RdBu_r', norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax), rasterized=True)
        cbar = fig.colorbar(mesh, ax=ax, pad=0.02, fraction=0.046)
        cbar.set_label(r'$\Delta\chi^2_{\mathrm{data}} = \chi^2(\mu^{\mathrm{spline}}) - \chi^2(\mu^{\mathrm{exact}})$')
        cbar.ax.tick_params(which='both', direction='in')
        draw_contours(ax, out['samples'][:, [i, j]], float(grid['axis'].max()))
        ax.axhline(0, color='0.55', lw=0.7)
        ax.axvline(0, color='0.55', lw=0.7)
        ax.set_xlim(edges[0], edges[-1])
        ax.set_ylim(edges[0], edges[-1])
        ax.set_aspect('equal')
        ax.set_xlabel(rf'$\eta_{i + 1}$ (standardized PCA coordinate, $\sqrt{{\lambda_{i + 1}}}$ = {group["sqrt_lambda"][i]:.3f})')
        ax.set_ylabel(rf'$\eta_{j + 1}$ (standardized PCA coordinate, $\sqrt{{\lambda_{j + 1}}}$ = {group["sqrt_lambda"][j]:.3f})')
        row = SUMMARY[(SUMMARY['fit'] == spec_key) & (SUMMARY['suite'] == SUITE_XML[suite]) & (SUMMARY['grid'] == grid['kind'])].iloc[0]
        ax.text(0.03, 0.97, f'{prior_label(spec)}\n{SUITE_LABEL[suite]}\n'
                rf'$\max|\Delta\chi^2|$: {row["dchi2_b1|68"]:.1e} (68%), {row["dchi2_b1|95"]:.1e} (95%), {row["dchi2_b1|grid"]:.1e} (grid)',
                transform=ax.transAxes, ha='left', va='top', fontsize=10.5, color='black',
                bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', pad=3))
        handles = [Line2D([], [], color='black', ls='-', lw=1.3, label='68% credible region (HPD)'),
                   Line2D([], [], color='black', ls='--', lw=1.3, label='95% credible region (HPD)'),
                   Line2D([], [], color=COLOR_GUIDE, marker='+', ls='none', markersize=9, markeredgewidth=1.6, label='posterior mean')]
        ax.legend(handles=handles, loc='lower right', fontsize=10, frameon=True, framealpha=0.85)
        ax.tick_params(which='both', direction='in', top=True, right=True)
        if SAVE_OUTPUTS:
            fig.savefig(FIG_DIR / f'{stem}.pdf', dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
            print('Saved:', FIG_DIR / f'{stem}.pdf')
        display(fig)
        plt.close(fig)


candidates = SUMMARY[(SUMMARY['region'] != 'no chain') & SUMMARY['grid'].str.startswith('full') & ~SUMMARY['fit'].str.contains('uniform') & (SUMMARY['n free'] == 2)]
worst = candidates.iloc[candidates['dchi2_b1|95'].argmax()]
print('Worst Gaussian-prior 2-parameter case by |dchi2| inside the 95% region:', worst['fit'], worst['suite'], f"{worst['dchi2_b1|95']:.2e}")
suite_key = next(s for s in SUITES if SUITE_XML[s] == worst['suite'])
summary_figure(worst['fit'], suite_key, 'full', 'spline_factorization_worst_case')
uniform = SUMMARY[(SUMMARY['region'] != 'no chain') & SUMMARY['grid'].str.startswith('extended')]
worst_u = uniform.iloc[uniform['dchi2_b1|95'].argmax()]
print('Worst uniform-prior case (extended grid) by |dchi2| inside the 95% region:', worst_u['fit'], worst_u['suite'], f"{worst_u['dchi2_b1|95']:.2e}")
summary_figure(worst_u['fit'], next(s for s in SUITES if SUITE_XML[s] == worst_u['suite']), 'extended', 'spline_factorization_worst_case_uniform')

## Save the $\Delta\chi^2$ grids for chain reweighting

One `.npz` per fit, suite and grid in `tables/spline_factorization/grids/`, holding `eta_axes`, `dchi2` (definition b1), `dchi2_b2`, `chi2_exact`, `max_frac` and the provenance. `spline_reweighting.py` turns them into importance weights $w_k=\exp(+\Delta\chi^2_{\rm data}(\eta^{(k)})/2)$ for the stored chains; the last cell is the round trip through that module.

In [ ]:
GRID_OUT_DIR = TABLE_DIR / 'grids'
GRID_OUT_DIR.mkdir(parents=True, exist_ok=True)
saved_rows = []
for spec in ZEXP_FITS:
    group = PRIOR_GROUPS[spec.prior.name]
    n = group['n_free']
    eta_cols = [f'eta{k + 1}' for k in range(n)]
    for suite in SUITES:
        out = FIT_OUTPUTS.get((spec.key, suite))
        for grid_id in ('full', 'extended'):
            frame = RESULTS.get((spec.prior.name, grid_id, suite))
            if frame is None:
                continue
            grid = GRIDS[(spec.prior.name, grid_id)]
            axis = np.asarray(grid['axis'], float)
            shape = (len(axis),) * n
            ordered = frame.sort_values(eta_cols, kind='mergesort')
            assert len(ordered) == len(axis) ** n, (spec.key, suite, grid_id, len(ordered))
            # eta1 is the slowest index ('ij' layout), which is what RegularGridInterpolator expects.
            mesh = np.meshgrid(*([axis] * n), indexing='ij')
            for k in range(n):
                assert np.allclose(ordered[eta_cols[k]].to_numpy().reshape(shape), mesh[k]), (spec.key, suite, grid_id, k)
            arrays = {col: ordered[col].to_numpy(float).reshape(shape) for col in ('dchi2_b1', 'dchi2_b2', 'chi2_exact', 'max_frac')}
            path = GRID_OUT_DIR / f'{spec.key}_{SUITE_XML[suite]}_{grid_id}_dchi2.npz'
            np.savez(
                path,
                eta_axes=np.array([axis] * n), dchi2=arrays['dchi2_b1'], dchi2_b2=arrays['dchi2_b2'],
                chi2_exact=arrays['chi2_exact'], max_frac=arrays['max_frac'],
                fit_name=spec.key, suite=SUITE_XML[suite], grid_type=grid_id, grid_kind=grid['kind'],
                prior=spec.prior.name, kmax=spec.prior.kmax, n_pca=n,
                eta_names=np.array(spec.prior.variation_branches),
                step=float(axis[1] - axis[0]), half_range=float(axis.max()),
                uniform_prior=bool(spec.uniform_prior), has_chain=out is not None,
                chain_production=out['chain_source'] if out is not None else 'none',
                n_chain_samples=len(out['samples']) if out is not None else 0,
                F_source=out['F_source'] if out is not None else 'none',
                data_source=DATA_SOURCE, dipole_fa_q2_zero=DIPOLE_FA_Q2_ZERO_IN_FILE, input_file=str(INPUT_FILE),
            )
            saved_rows.append({'fit': spec.key, 'suite': SUITE_XML[suite], 'grid': grid_id, 'shape': str(shape),
                               'step': float(axis[1] - axis[0]), 'half-range': float(axis.max()),
                               'has chain': out is not None, 'max |dchi2| on grid': float(np.abs(arrays['dchi2_b1']).max()),
                               'file': path.name})
saved_grids = pd.DataFrame(saved_rows)
print(f'{len(saved_grids)} grids written to {GRID_OUT_DIR} ({int(saved_grids["has chain"].sum())} with chain data); '
      f'largest |dchi2| on any grid {saved_grids["max |dchi2| on grid"].max():.3g}.')

In [ ]:
import warnings
import importlib
import spline_reweighting
importlib.reload(spline_reweighting)
from spline_reweighting import compute_importance_weights, load_dchi2_grid

check_rows = []
for spec in ZEXP_FITS:
    for suite in SUITES:
        out = FIT_OUTPUTS.get((spec.key, suite))
        if out is None:
            continue
        grid = load_dchi2_grid(spec.key, SUITE_XML[suite])      # 'auto': extended for uniform-prior fits, full otherwise
        assert tuple(grid['eta_names']) == tuple(spec.prior.variation_branches)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            w_raw, w, ess, dchi2 = compute_importance_weights(out['samples'], grid)
        samples = out['samples']
        mean0, std0 = samples.mean(axis=0), samples.std(axis=0)
        mean1 = np.average(samples, weights=w, axis=0)
        std1 = np.sqrt(np.average((samples - mean1) ** 2, weights=w, axis=0))
        row = SUMMARY[(SUMMARY['fit'] == spec.key) & (SUMMARY['suite'] == SUITE_XML[suite]) & (SUMMARY['grid'] == grid['grid_kind'])].iloc[0]
        check_rows.append({'fit': spec.key, 'suite': SUITE_XML[suite], 'grid': grid['grid_type'], 'N': len(samples), 'ESS / N': ess / len(samples),
                           'max |dchi2| at samples': np.abs(dchi2).max(),
                           'mean shift / sigma (module)': np.max(np.abs(mean1 - mean0) / std0),
                           'mean shift / sigma (table)': row['max |mean shift| / sigma'],
                           '|sigma ratio - 1| (module)': np.max(np.abs(std1 / std0 - 1)),
                           '|sigma ratio - 1| (table)': row['max |sigma ratio - 1|'],
                           'module warnings': '; '.join(str(c.message).split(': ', 1)[-1][:70] for c in caught) or '-'})
roundtrip = pd.DataFrame(check_rows)
assert np.allclose(roundtrip['mean shift / sigma (module)'], roundtrip['mean shift / sigma (table)'], atol=1e-6, rtol=1e-3)
assert np.allclose(roundtrip['|sigma ratio - 1| (module)'], roundtrip['|sigma ratio - 1| (table)'], atol=1e-6, rtol=1e-3)
gaussian_rows = roundtrip[~roundtrip['fit'].str.contains('uniform')]
print(f"Gaussian-prior fits: ESS/N >= {gaussian_rows['ESS / N'].min():.4f}, max |dchi2| at the samples {gaussian_rows['max |dchi2| at samples'].max():.3g}, "
      f"mean shift <= {gaussian_rows['mean shift / sigma (module)'].max():.4f} sigma.")
uniform_rows = roundtrip[roundtrip['fit'].str.contains('uniform')]
print(f"Uniform-prior fits: ESS/N >= {uniform_rows['ESS / N'].min():.3f}, mean shift up to {uniform_rows['mean shift / sigma (module)'].max():.3f} sigma, "
      f"width change up to {100 * uniform_rows['|sigma ratio - 1| (module)'].max():.1f}%.")
print('The saved grids and spline_reweighting.compute_importance_weights reproduce the summary-table reweighting.')